# CopulaAlpha LAB - Mixture Copula Only Research Notebook

Questo notebook e una copia semplificata del progetto **Copula-Based Relative-Value Alpha Engine**, focalizzata solo sulla **mixture copula**.

L'obiettivo e costruire una pipeline end-to-end, semplice ma completa, per testare una **mixture copula** composta da Gaussian, Student-t e Clayton come segnale relative-value tra ETF liquidi.

Workflow MVP:

1. scaricare adjusted close ETF;
2. calcolare log returns giornalieri;
3. selezionare coppie top-correlated usando solo il periodo train;
4. stimare una mixture copula rolling per ogni coppia;
5. calcolare due mispricing index condizionali e combinarli in uno score bidirezionale;
6. costruire un portafoglio pair-trading dollar-neutral con soglie per-coppia ed exit esplicite;
7. backtestare con shift di un giorno e transaction costs;
8. calcolare metriche di performance e IC.

Nota importante: in questo MVP non assumiamo che la copula migliori la strategia. La testiamo come feature di alpha research.


## Domanda di ricerca

La domanda principale e:

> Una mixture copula rolling riesce a estrarre segnali relative-value stabili tra ETF correlati?

Il criterio sperimentale e out-of-sample:

- il pair universe viene selezionato sul train;
- i parametri rolling usano solo dati passati;
- le posizioni sono shiftate di un giorno prima del PnL;
- i risultati vengono separati in train, validation e test.

Split usato nel notebook:

| Periodo | Date |
|---|---|
| Train | 2015-01-01 / 2020-12-31 |
| Validation | 2021-01-01 / 2022-12-31 |
| Test | 2023-01-01 / latest available date |

## Matematica del segnale

I prezzi adjusted close vengono trasformati in log returns:

$$
r_{i,t} = \log\left(\frac{P_{i,t}}{P_{i,t-1}}\right)
$$

Per ogni coppia di asset, i returns nella finestra rolling di lunghezza $W$ vengono trasformati in pseudo-osservazioni empiriche:

$$
u_{i,t} = \frac{\operatorname{rank}(r_{i,t})}{W+1}
$$

### Gaussian copula

La Gaussian copula bivariata e:

$$
C_{\rho}(u,v) = \Phi_{\rho}\left(\Phi^{-1}(u), \Phi^{-1}(v)\right)
$$

dove $\Phi^{-1}$ e la inverse standard normal CDF, $\Phi_{\rho}$ e la bivariate normal CDF, e $\rho$ e il parametro di dipendenza.

Definiamo:

$$
z_i = \Phi^{-1}(u_i), \qquad z_j = \Phi^{-1}(u_j)
$$

Per una Gaussian copula:

$$
z_i \mid z_j \sim \mathcal{N}\left(\rho z_j, 1 - \rho^2\right)
$$

La probabilita condizionale osservata e:

$$
p_{i \mid j,t} = \Phi\left(\frac{z_{i,t} - \rho_t z_{j,t}}{\sqrt{1 - \rho_t^2}}\right)
$$

Uno score direzionale semplice sarebbe:

$$
s_{i \mid j,t} = p_{i \mid j,t} - 0.5
$$

Interpretazione della singola probabilita condizionale:

- $s_{i \mid j,t} > 0$: asset $i$ appare rich rispetto a $j$;
- $s_{i \mid j,t} < 0$: asset $i$ appare cheap rispetto a $j$.

Nel portfolio finale, pero, non usiamo piu una sola probabilita condizionale. Usiamo due mispricing index simmetrici, descritti sotto.

### Student-t copula

La Student-t copula sostituisce la normale multivariata con una t multivariata. Questo permette di catturare co-movimenti estremi simmetrici, utili quando due asset diventano piu dipendenti nelle code.

$$
C_{\rho,\nu}^{t}(u,v) = t_{\rho,\nu}\left(t_{\nu}^{-1}(u), t_{\nu}^{-1}(v)\right)
$$

dove $t_{\nu}^{-1}$ e la inverse CDF della t univariata con $\nu$ gradi di liberta, mentre $t_{\rho,\nu}$ e la CDF t bivariata con correlazione $\rho$.

Definiamo:

$$
x_i = t_{\nu}^{-1}(u_i), \qquad x_j = t_{\nu}^{-1}(u_j)
$$

Per una t bivariata:

$$
x_i \mid x_j \sim t_{\nu+1}\left(\rho x_j, \sqrt{\frac{(\nu + x_j^2)(1 - \rho^2)}{\nu + 1}}\right)
$$

La probabilita condizionale usata nel segnale e:

$$
p_{i \mid j,t}^{t} = t_{\nu+1}\left(\frac{x_{i,t} - \rho_t x_{j,t}}{\sqrt{\frac{(\nu + x_{j,t}^{2})(1 - \rho_t^2)}{\nu + 1}}}\right)
$$

Nel MVP $\nu$ e fissato da configurazione. La stima rolling di $\nu$ e lasciata come estensione, perche puo diventare instabile su finestre corte.

### Clayton copula

La Clayton copula e una copula Archimedean con lower-tail dependence. E utile quando la dipendenza aumenta soprattutto nei movimenti negativi o risk-off.

$$
C_{\theta}^{Clayton}(u,v) = \left(u^{-\theta} + v^{-\theta} - 1\right)^{-1/\theta}, \qquad \theta > 0
$$

La relazione tra Kendall tau e il parametro Clayton e:

$$
\tau = \frac{\theta}{\theta + 2}
$$

quindi:

$$
\theta = \frac{2\tau}{1 - \tau}
$$

La conditional CDF di $U_i$ dato $U_j=v$ e la derivata parziale della copula rispetto a $v$:

$$
p_{i \mid j,t}^{Clayton} = \frac{\partial C_{\theta}(u_i,u_j)}{\partial u_j}
$$

$$
p_{i \mid j,t}^{Clayton} = u_{j,t}^{-\theta-1}\left(u_{i,t}^{-\theta} + u_{j,t}^{-\theta} - 1\right)^{-1/\theta - 1}
$$

La Clayton copula classica supporta dipendenza positiva. Se la Kendall tau rolling e non positiva, il codice usa il limite di indipendenza:

$$
p_{i \mid j,t}^{independent} = u_{i,t}
$$

### Mixture copula

La mixture copula combina piu famiglie di copule in un unico modello pesato. Nel notebook usiamo una mixture di Gaussian, Student-t e Clayton:

$$
C_{mix}(u,v) = w_G C_G(u,v) + w_T C_T(u,v) + w_C C_C(u,v)
$$

con vincoli:

$$
w_G \ge 0, \qquad w_T \ge 0, \qquad w_C \ge 0
$$

$$
w_G + w_T + w_C = 1
$$

La probabilita condizionale della mixture e una combinazione pesata delle probabilita condizionali dei componenti:

$$
p_{i \mid j,t}^{mix} = w_G p_{i \mid j,t}^{G} + w_T p_{i \mid j,t}^{T} + w_C p_{i \mid j,t}^{C}
$$

### Stima dei pesi mixture

Invece di fissare sempre $w_G=w_T=w_C=1/3$, possiamo stimare i pesi sulla finestra rolling. Usiamo la likelihood di copula dei dati storici nella finestra:

$$
c_{mix}(u,v) = w_G c_G(u,v) + w_T c_T(u,v) + w_C c_C(u,v)
$$

Per ogni data $t$, i pesi vengono stimati usando solo la finestra passata:

$$
\widehat{w}_t = \arg\max_{w_G,w_T,w_C}\sum_{s=t-W}^{t-1}\log\left(w_G c_G(u_{i,s},u_{j,s}) + w_T c_T(u_{i,s},u_{j,s}) + w_C c_C(u_{i,s},u_{j,s})\right)
$$

soggetto a:

$$
w_G+w_T+w_C=1
$$

Nel codice la massimizzazione sui soli pesi viene fatta con un EM leggero e un floor minimo sui pesi, cosi nessuna componente collassa completamente a zero.

### Mispricing index bidirezionale

Per rendere il segnale piu coerente con il pair trading, calcoliamo entrambe le conditional probabilities della coppia:

$$
MI_{i \mid j,t} = P(U_i \le u_{i,t} \mid U_j = u_{j,t})
$$

$$
MI_{j \mid i,t} = P(U_j \le u_{j,t} \mid U_i = u_{i,t})
$$

Lo score finale e la differenza normalizzata tra i due mispricing index:

$$
S_{ij,t} = \frac{MI_{i \mid j,t} - MI_{j \mid i,t}}{2}
$$

Interpretazione operativa:

- $S_{ij,t} > 0$: asset $i$ appare rich e asset $j$ cheap, quindi short $i$, long $j$;
- $S_{ij,t} < 0$: asset $i$ appare cheap e asset $j$ rich, quindi long $i$, short $j$.

L'entry non usa piu una soglia globale fissa. Per ogni coppia stimiamo sul train una soglia percentile:

$$
q_{ij} = \max\left(q_{min}, Q_{\alpha}\left(|S_{ij,t}|\right)\right)
$$

La coppia puo entrare solo quando:

$$
|S_{ij,t}| \ge q_{ij}
$$

Questo evita di trattare allo stesso modo coppie naturalmente rumorose e coppie piu stabili.


In [ ]:
from __future__ import annotations

import math
import warnings
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import gammaln
from scipy.stats import kendalltau, norm, rankdata, spearmanr, t as student_t

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = PROJECT_ROOT / "results" / "mixture_only"
PLOTS_DIR = RESULTS_DIR / "plots"

for directory in [DATA_RAW_DIR, DATA_PROCESSED_DIR, RESULTS_DIR, PLOTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

ETF_UNIVERSE = [
    "SPY", "QQQ", "IWM", "DIA",
    "EFA", "EEM", "VGK", "EWJ",
    "TLT", "IEF", "SHY", "LQD", "HYG",
    "GLD", "SLV", "USO", "DBA",
    "XLK", "XLF", "XLE", "XLY", "XLI",
    "XLP", "XLU", "XLV", "XLB", "XLRE",
    "SMH", "IBB", "KRE", "XRT", "ITA",
    "VNQ", "IYR", "DBC", "UUP", "FXE",
    "FXY", "FXB", "EWT", "EWG", "EWU",
]

@dataclass(frozen=True)
class MVPConfig:
    start_date: str = "2015-01-01"
    end_date: str | None = None
    train_end: str = "2020-12-31"
    validation_start: str = "2021-01-01"
    validation_end: str = "2022-12-31"
    test_start: str = "2023-01-01"
    rolling_window: int = 252
    copula_model: str = "mixture"
    student_t_df: int = 6
    mixture_weights: tuple[float, float, float] = (1.0 / 3.0, 1.0 / 3.0, 1.0 / 3.0)
    estimate_mixture_weights: bool = False
    mixture_weight_floor: float = 0.05
    mixture_weight_max_iter: int = 50
    mixture_weight_tol: float = 1e-6
    peers_per_asset: int = 8
    max_candidate_pairs: int = 20
    excluded_assets: tuple[str, ...] = ("SPY", "DIA", "QQQ")
    blocked_overlap_groups: tuple[tuple[str, ...], ...] = (
        ("EFA", "VGK", "EWG", "EWU"),
        ("VNQ", "IYR", "XLRE"),
        ("DBC", "USO", "DBA"),
    )
    min_abs_corr: float = 0.60
    max_spread_half_life: float = 1.00
    min_spread_vol: float = 0.004
    top_n_pairs: int = 1
    entry_quantile: float = 0.90
    exit_quantile: float = 0.50
    min_entry_score: float = 0.25
    max_holding_days: int = 1
    expected_edge_cost_multiplier: float = 0.50
    min_expected_edge_bps: float = 2.0
    min_expected_edge_observations: int = 20
    rebalance_every_n_days: int = 5
    max_daily_turnover: float | None = 0.35
    cost_bps: float = 2.5
    min_history_fraction: float = 0.85

CONFIG = MVPConfig()
CONFIG


## Data pipeline

La funzione sotto scarica adjusted close via `yfinance`, salva una cache locale e calcola log returns. Se `yfinance` non e installato, installarlo nell'ambiente del notebook:

```bash
pip install yfinance
```

La pulizia MVP e intenzionalmente semplice: rimuove asset con storia insufficiente, forward-fill limitato dal dataset scaricato, poi tiene solo date comuni agli asset rimasti.

In [ ]:
def download_adjusted_prices(tickers: list[str], start: str, end: str | None = None) -> pd.DataFrame:
    try:
        import yfinance as yf
    except ImportError as exc:
        raise ImportError("yfinance non e installato. Esegui: pip install yfinance") from exc

    raw = yf.download(
        tickers,
        start=start,
        end=end,
        auto_adjust=True,
        progress=False,
        group_by="column",
        threads=True,
    )

    if isinstance(raw.columns, pd.MultiIndex):
        if "Close" not in raw.columns.get_level_values(0):
            raise ValueError("Colonna Close non trovata nel download yfinance.")
        prices = raw["Close"].copy()
    else:
        prices = raw[["Close"]].copy()
        prices.columns = tickers[:1]

    prices = prices.sort_index()
    prices.index = pd.to_datetime(prices.index)
    prices = prices.apply(pd.to_numeric, errors="coerce")
    prices = prices.dropna(axis=1, how="all")

    min_obs = int(len(prices) * CONFIG.min_history_fraction)
    prices = prices.dropna(axis=1, thresh=min_obs)
    prices = prices.ffill().dropna(axis=0, how="any")
    prices = prices.loc[:, sorted(prices.columns)]
    return prices


def load_or_download_prices(refresh: bool = False) -> pd.DataFrame:
    prices_path = DATA_PROCESSED_DIR / "prices.csv"
    if prices_path.exists() and not refresh:
        prices = pd.read_csv(prices_path, index_col=0, parse_dates=True)
        return prices.sort_index()

    prices = download_adjusted_prices(ETF_UNIVERSE, CONFIG.start_date, CONFIG.end_date)
    prices.to_csv(prices_path)
    return prices


def compute_log_returns(prices: pd.DataFrame) -> pd.DataFrame:
    returns = np.log(prices / prices.shift(1)).dropna(how="any")
    returns.to_csv(DATA_PROCESSED_DIR / "returns.csv")
    return returns

## Pair selection

Per evitare look-ahead bias, le coppie candidate vengono selezionate solo sul periodo train.

La vecchia regola era solo top-correlation. Ora usiamo una selezione piu severa:

1. rimuoviamo ETF troppo efficienti o benchmark molto sovrapposti, per esempio `SPY`, `DIA`, `QQQ`;
2. blocchiamo coppie chiaramente duplicate dentro alcuni gruppi di overlap;
3. richiediamo alta correlazione dei returns;
4. richiediamo mean reversion dello spread residuale dei returns.

Lo spread residuale usato nel filtro e:

$$
\varepsilon_{ij,t} = r_{i,t} - \beta_{ij} r_{j,t}
$$

con:

$$
\beta_{ij} = \frac{\operatorname{Cov}(r_i,r_j)}{\operatorname{Var}(r_j)}
$$

Poi stimiamo una half-life AR(1) sullo spread residuale. La coppia viene tenuta solo se lo spread e abbastanza volatile e mean-reverting sul train.

Questa non e cointegration sui prezzi. E un filtro short-term coerente con un segnale copula sui returns.


In [ ]:
def is_blocked_overlap_pair(
    asset_i: str,
    asset_j: str,
    blocked_overlap_groups: tuple[tuple[str, ...], ...],
) -> bool:
    return any(asset_i in group and asset_j in group for group in blocked_overlap_groups)


def estimate_hedge_beta(x: pd.Series, y: pd.Series) -> float:
    y_var = float(y.var(ddof=1))
    if y_var <= 0 or not np.isfinite(y_var):
        return np.nan
    return float(x.cov(y) / y_var)


def estimate_half_life(series: pd.Series) -> float:
    clean = series.dropna()
    if len(clean) < 50:
        return np.inf

    lagged = clean.shift(1).dropna()
    delta = clean.diff().dropna()
    lagged = lagged.loc[delta.index]

    lagged_centered = lagged - lagged.mean()
    delta_centered = delta - delta.mean()
    denominator = float((lagged_centered**2).sum())
    if denominator <= 0:
        return np.inf

    lambda_hat = float((lagged_centered * delta_centered).sum() / denominator)
    if lambda_hat >= 0 or not np.isfinite(lambda_hat):
        return np.inf
    return float(-math.log(2.0) / lambda_hat)


def select_candidate_pairs(
    returns: pd.DataFrame,
    train_end: str,
    top_k: int,
    max_pairs: int,
    excluded_assets: tuple[str, ...] = (),
    blocked_overlap_groups: tuple[tuple[str, ...], ...] = (),
    min_abs_corr: float = 0.60,
    max_spread_half_life: float = 1.00,
    min_spread_vol: float = 0.004,
) -> pd.DataFrame:
    train_returns = returns.loc[:train_end].copy()
    excluded = set(excluded_assets)
    tradable_columns = [col for col in train_returns.columns if col not in excluded]
    train_returns = train_returns[tradable_columns].dropna(how="any")
    corr = train_returns.corr()

    pair_records: list[dict[str, object]] = []
    seen: set[tuple[str, str]] = set()

    for asset in corr.columns:
        peers = corr[asset].drop(labels=[asset]).sort_values(ascending=False).head(top_k)
        for peer, rho in peers.items():
            pair = tuple(sorted((asset, peer)))
            if pair in seen:
                continue
            seen.add(pair)

            asset_i, asset_j = pair
            if is_blocked_overlap_pair(asset_i, asset_j, blocked_overlap_groups):
                continue
            if not np.isfinite(rho) or rho < min_abs_corr:
                continue

            pair_returns = train_returns[[asset_i, asset_j]].dropna()
            if len(pair_returns) < 252:
                continue

            beta = estimate_hedge_beta(pair_returns[asset_i], pair_returns[asset_j])
            if not np.isfinite(beta):
                continue

            spread = pair_returns[asset_i] - beta * pair_returns[asset_j]
            spread_half_life = estimate_half_life(spread)
            spread_vol = float(spread.std(ddof=1))
            spread_lag1_autocorr = float(spread.autocorr(1))

            if not np.isfinite(spread_half_life) or spread_half_life > max_spread_half_life:
                continue
            if not np.isfinite(spread_vol) or spread_vol < min_spread_vol:
                continue

            pair_records.append(
                {
                    "asset_i": asset_i,
                    "asset_j": asset_j,
                    "corr_train": float(corr.loc[asset_i, asset_j]),
                    "beta_train": beta,
                    "spread_half_life": spread_half_life,
                    "spread_vol": spread_vol,
                    "spread_lag1_autocorr": spread_lag1_autocorr,
                }
            )

    pairs = pd.DataFrame(pair_records)
    if pairs.empty:
        raise ValueError("No candidate pairs survived pair selection v2. Relax min_abs_corr, max_spread_half_life, min_spread_vol, or overlap filters.")

    pairs["corr_rank"] = pairs["corr_train"].rank(pct=True)
    pairs["mean_reversion_rank"] = (-pairs["spread_half_life"]).rank(pct=True)
    pairs["spread_vol_rank"] = pairs["spread_vol"].rank(pct=True)
    pairs["pair_selection_score"] = (
        0.50 * pairs["corr_rank"]
        + 0.35 * pairs["mean_reversion_rank"]
        + 0.15 * pairs["spread_vol_rank"]
    )

    pairs = pairs.sort_values("pair_selection_score", ascending=False).head(max_pairs).reset_index(drop=True)
    pairs.to_csv(DATA_PROCESSED_DIR / "candidate_pairs.csv", index=False)
    return pairs


def describe_pairs(pairs: pd.DataFrame) -> pd.DataFrame:
    numeric_cols = [
        "corr_train",
        "beta_train",
        "spread_half_life",
        "spread_vol",
        "spread_lag1_autocorr",
        "pair_selection_score",
    ]
    return pairs[numeric_cols].describe().T


## Rolling copula signals

Per ogni data $t$, stimiamo i parametri della mixture copula sulla finestra $t-W$ fino a $t-1$, poi valutiamo il return osservato in $t$. Questo permette di generare il segnale a fine giornata $t$ senza usare dati futuri. Il PnL usa posizioni shiftate di un giorno.

La stima di $\rho_t$ per Gaussian e Student-t avviene sui transformed scores dei ranks empirici della finestra rolling. Per Clayton stimiamo $\theta_t$ dalla Kendall tau rolling.

Di default usiamo pesi fissi `1/3, 1/3, 1/3`, che nel test precedente sono risultati piu robusti. La stima rolling dei pesi resta disponibile impostando `estimate_mixture_weights = True`, ma non e la configurazione base.

Per ogni coppia salviamo sia $MI_{i \mid j,t}$ sia $MI_{j \mid i,t}$. Lo score tradabile e `bidirectional_score`, uguale a $S_{ij,t}$.


In [ ]:
SUPPORTED_COPULA_MODELS = {"mixture", "mixed"}
COPULA_MODEL_ALIASES = {"mixed": "mixture"}
MIN_COPULA_DENSITY = 1e-12
MAX_COPULA_DENSITY = 1e12


def canonical_copula_model(copula_model: str) -> str:
    copula_model = copula_model.strip().lower()
    return COPULA_MODEL_ALIASES.get(copula_model, copula_model)


def normalize_mixture_weights(weights: tuple[float, float, float]) -> tuple[float, float, float]:
    weight_array = np.asarray(weights, dtype=float)
    if weight_array.shape != (3,):
        raise ValueError("mixture_weights must contain exactly three values: Gaussian, Student-t, Clayton.")
    if np.any(weight_array < 0):
        raise ValueError("mixture_weights must be non-negative.")
    total_weight = weight_array.sum()
    if total_weight <= 0:
        raise ValueError("At least one mixture weight must be positive.")
    normalized = weight_array / total_weight
    return float(normalized[0]), float(normalized[1]), float(normalized[2])


def empirical_u_history(values: np.ndarray) -> np.ndarray:
    ranks = rankdata(values, method="average")
    return ranks / (len(values) + 1.0)


def empirical_u_current(history: np.ndarray, current_value: float) -> float:
    # Smoothed empirical CDF to avoid exact 0 or 1.
    return (np.sum(history <= current_value) + 1.0) / (len(history) + 2.0)


def estimate_gaussian_copula_rho(x_hist: np.ndarray, y_hist: np.ndarray) -> float:
    ux = empirical_u_history(x_hist)
    uy = empirical_u_history(y_hist)
    zx = norm.ppf(ux)
    zy = norm.ppf(uy)
    rho = np.corrcoef(zx, zy)[0, 1]
    if np.isnan(rho):
        return 0.0
    return float(np.clip(rho, -0.95, 0.95))


def estimate_student_t_copula_rho(x_hist: np.ndarray, y_hist: np.ndarray, df: int) -> float:
    ux = empirical_u_history(x_hist)
    uy = empirical_u_history(y_hist)
    tx = student_t.ppf(ux, df=df)
    ty = student_t.ppf(uy, df=df)
    rho = np.corrcoef(tx, ty)[0, 1]
    if np.isnan(rho):
        return 0.0
    return float(np.clip(rho, -0.95, 0.95))


def estimate_clayton_theta(x_hist: np.ndarray, y_hist: np.ndarray) -> tuple[float, float]:
    tau_result = kendalltau(x_hist, y_hist, nan_policy="omit")
    tau = float(tau_result.statistic if hasattr(tau_result, "statistic") else tau_result[0])
    if np.isnan(tau) or tau <= 0:
        return 0.0, tau
    tau = float(np.clip(tau, 1e-6, 0.95))
    theta = 2.0 * tau / (1.0 - tau)
    return float(np.clip(theta, 1e-6, 30.0)), tau


def gaussian_conditional_probability(u_i: float, u_j: float, rho: float) -> float:
    z_i = norm.ppf(np.clip(u_i, 1e-6, 1 - 1e-6))
    z_j = norm.ppf(np.clip(u_j, 1e-6, 1 - 1e-6))
    denominator = math.sqrt(max(1e-8, 1.0 - rho**2))
    return float(norm.cdf((z_i - rho * z_j) / denominator))


def student_t_conditional_probability(u_i: float, u_j: float, rho: float, df: int) -> float:
    x_i = student_t.ppf(np.clip(u_i, 1e-6, 1 - 1e-6), df=df)
    x_j = student_t.ppf(np.clip(u_j, 1e-6, 1 - 1e-6), df=df)
    scale = math.sqrt(max(1e-8, ((df + x_j**2) * (1.0 - rho**2)) / (df + 1.0)))
    return float(student_t.cdf((x_i - rho * x_j) / scale, df=df + 1))


def clayton_conditional_probability(u_i: float, u_j: float, theta: float) -> float:
    u_i = float(np.clip(u_i, 1e-6, 1 - 1e-6))
    u_j = float(np.clip(u_j, 1e-6, 1 - 1e-6))
    if theta <= 1e-6:
        return u_i
    base = max(u_i ** (-theta) + u_j ** (-theta) - 1.0, 1e-12)
    conditional_prob = u_j ** (-theta - 1.0) * base ** (-1.0 / theta - 1.0)
    return float(np.clip(conditional_prob, 0.0, 1.0))


def _clip_density_from_log(log_density: np.ndarray) -> np.ndarray:
    clipped_log = np.clip(log_density, math.log(MIN_COPULA_DENSITY), math.log(MAX_COPULA_DENSITY))
    return np.clip(np.exp(clipped_log), MIN_COPULA_DENSITY, MAX_COPULA_DENSITY)


def gaussian_copula_density(u_i: np.ndarray, u_j: np.ndarray, rho: float) -> np.ndarray:
    u_i = np.asarray(np.clip(u_i, 1e-6, 1 - 1e-6), dtype=float)
    u_j = np.asarray(np.clip(u_j, 1e-6, 1 - 1e-6), dtype=float)
    z_i = norm.ppf(u_i)
    z_j = norm.ppf(u_j)
    rho = float(np.clip(rho, -0.95, 0.95))
    one_minus_rho2 = max(1e-8, 1.0 - rho**2)
    log_density = (
        -0.5 * math.log(one_minus_rho2)
        - (z_i**2 - 2.0 * rho * z_i * z_j + z_j**2) / (2.0 * one_minus_rho2)
        + 0.5 * (z_i**2 + z_j**2)
    )
    return _clip_density_from_log(log_density)


def student_t_copula_density(u_i: np.ndarray, u_j: np.ndarray, rho: float, df: int) -> np.ndarray:
    u_i = np.asarray(np.clip(u_i, 1e-6, 1 - 1e-6), dtype=float)
    u_j = np.asarray(np.clip(u_j, 1e-6, 1 - 1e-6), dtype=float)
    x_i = student_t.ppf(u_i, df=df)
    x_j = student_t.ppf(u_j, df=df)
    rho = float(np.clip(rho, -0.95, 0.95))
    determinant = max(1e-8, 1.0 - rho**2)
    quadratic = (x_i**2 - 2.0 * rho * x_i * x_j + x_j**2) / determinant
    log_bivariate_t = (
        gammaln((df + 2.0) / 2.0)
        - gammaln(df / 2.0)
        - math.log(df * math.pi)
        - 0.5 * math.log(determinant)
        - ((df + 2.0) / 2.0) * np.log1p(quadratic / df)
    )
    log_univariate_t = student_t.logpdf(x_i, df=df) + student_t.logpdf(x_j, df=df)
    return _clip_density_from_log(log_bivariate_t - log_univariate_t)


def clayton_copula_density(u_i: np.ndarray, u_j: np.ndarray, theta: float) -> np.ndarray:
    u_i = np.asarray(np.clip(u_i, 1e-6, 1 - 1e-6), dtype=float)
    u_j = np.asarray(np.clip(u_j, 1e-6, 1 - 1e-6), dtype=float)
    if theta <= 1e-6:
        return np.ones(np.broadcast(u_i, u_j).shape)
    base = np.maximum(u_i ** (-theta) + u_j ** (-theta) - 1.0, 1e-12)
    log_density = (
        math.log1p(theta)
        + (-theta - 1.0) * (np.log(u_i) + np.log(u_j))
        + (-2.0 - 1.0 / theta) * np.log(base)
    )
    return _clip_density_from_log(log_density)


def mixture_component_density_matrix(
    x_hist: np.ndarray,
    y_hist: np.ndarray,
    params: dict[str, float],
    student_t_df: int,
) -> np.ndarray:
    u_hist_i = empirical_u_history(x_hist)
    u_hist_j = empirical_u_history(y_hist)
    density_matrix = np.column_stack(
        [
            gaussian_copula_density(u_hist_i, u_hist_j, params["rho"]),
            student_t_copula_density(u_hist_i, u_hist_j, params["student_t_rho"], df=student_t_df),
            clayton_copula_density(u_hist_i, u_hist_j, params["theta"]),
        ]
    )
    return np.clip(density_matrix, MIN_COPULA_DENSITY, MAX_COPULA_DENSITY)


def estimate_mixture_weights_by_likelihood(
    x_hist: np.ndarray,
    y_hist: np.ndarray,
    params: dict[str, float],
    student_t_df: int,
    initial_weights: tuple[float, float, float],
    weight_floor: float,
    max_iter: int,
    tol: float,
) -> tuple[tuple[float, float, float], float]:
    if weight_floor < 0 or weight_floor * 3.0 >= 1.0:
        raise ValueError("mixture_weight_floor must be non-negative and lower than 1/3.")
    if max_iter < 1:
        raise ValueError("mixture_weight_max_iter must be at least 1.")

    density_matrix = mixture_component_density_matrix(
        x_hist=x_hist,
        y_hist=y_hist,
        params=params,
        student_t_df=student_t_df,
    )
    if not np.isfinite(density_matrix).all():
        return normalize_mixture_weights(initial_weights), np.nan

    weights = np.asarray(normalize_mixture_weights(initial_weights), dtype=float)
    if weight_floor > 0:
        weights = np.maximum(weights, weight_floor)
        weights = weights / weights.sum()

    previous_log_likelihood = -np.inf
    for _ in range(max_iter):
        mixture_density = np.clip(density_matrix @ weights, MIN_COPULA_DENSITY, MAX_COPULA_DENSITY)
        responsibilities = density_matrix * weights / mixture_density[:, None]
        new_weights = responsibilities.mean(axis=0)
        if weight_floor > 0:
            new_weights = np.maximum(new_weights, weight_floor)
        new_weights = new_weights / new_weights.sum()

        new_mixture_density = np.clip(density_matrix @ new_weights, MIN_COPULA_DENSITY, MAX_COPULA_DENSITY)
        log_likelihood = float(np.mean(np.log(new_mixture_density)))
        if (
            abs(log_likelihood - previous_log_likelihood) < tol
            and float(np.max(np.abs(new_weights - weights))) < tol
        ):
            weights = new_weights
            previous_log_likelihood = log_likelihood
            break
        weights = new_weights
        previous_log_likelihood = log_likelihood

    final_density = np.clip(density_matrix @ weights, MIN_COPULA_DENSITY, MAX_COPULA_DENSITY)
    final_log_likelihood = float(np.mean(np.log(final_density)))
    return (float(weights[0]), float(weights[1]), float(weights[2])), final_log_likelihood


def estimate_mixture_component_params(
    x_hist: np.ndarray,
    y_hist: np.ndarray,
    student_t_df: int,
    mixture_weights: tuple[float, float, float],
    estimate_mixture_weights: bool = False,
    mixture_weight_floor: float = 0.05,
    mixture_weight_max_iter: int = 50,
    mixture_weight_tol: float = 1e-6,
) -> dict[str, float]:
    weight_gaussian, weight_student_t, weight_clayton = normalize_mixture_weights(mixture_weights)
    clayton_theta, clayton_tau = estimate_clayton_theta(x_hist, y_hist)
    params = {
        "rho": estimate_gaussian_copula_rho(x_hist, y_hist),
        "student_t_rho": estimate_student_t_copula_rho(x_hist, y_hist, df=student_t_df),
        "theta": clayton_theta,
        "kendall_tau": clayton_tau,
        "nu": float(student_t_df),
        "weight_gaussian": weight_gaussian,
        "weight_student_t": weight_student_t,
        "weight_clayton": weight_clayton,
        "mixture_log_likelihood": np.nan,
    }

    if estimate_mixture_weights:
        estimated_weights, log_likelihood = estimate_mixture_weights_by_likelihood(
            x_hist=x_hist,
            y_hist=y_hist,
            params=params,
            student_t_df=student_t_df,
            initial_weights=mixture_weights,
            weight_floor=mixture_weight_floor,
            max_iter=mixture_weight_max_iter,
            tol=mixture_weight_tol,
        )
        params["weight_gaussian"], params["weight_student_t"], params["weight_clayton"] = estimated_weights
        params["mixture_log_likelihood"] = log_likelihood

    return params


def mixture_conditional_probabilities(
    u_i: float,
    u_j: float,
    params: dict[str, float],
    student_t_df: int,
) -> dict[str, float]:
    g_i_given_j = gaussian_conditional_probability(u_i, u_j, params["rho"])
    t_i_given_j = student_t_conditional_probability(u_i, u_j, params["student_t_rho"], df=student_t_df)
    c_i_given_j = clayton_conditional_probability(u_i, u_j, params["theta"])

    g_j_given_i = gaussian_conditional_probability(u_j, u_i, params["rho"])
    t_j_given_i = student_t_conditional_probability(u_j, u_i, params["student_t_rho"], df=student_t_df)
    c_j_given_i = clayton_conditional_probability(u_j, u_i, params["theta"])

    mi_i_given_j = (
        params["weight_gaussian"] * g_i_given_j
        + params["weight_student_t"] * t_i_given_j
        + params["weight_clayton"] * c_i_given_j
    )
    mi_j_given_i = (
        params["weight_gaussian"] * g_j_given_i
        + params["weight_student_t"] * t_j_given_i
        + params["weight_clayton"] * c_j_given_i
    )

    return {
        "conditional_prob_gaussian": g_i_given_j,
        "conditional_prob_student_t": t_i_given_j,
        "conditional_prob_clayton": c_i_given_j,
        "conditional_prob_gaussian_reverse": g_j_given_i,
        "conditional_prob_student_t_reverse": t_j_given_i,
        "conditional_prob_clayton_reverse": c_j_given_i,
        "mi_i_given_j": float(mi_i_given_j),
        "mi_j_given_i": float(mi_j_given_i),
        "conditional_prob_i_given_j": float(mi_i_given_j),
        "conditional_prob_j_given_i": float(mi_j_given_i),
    }


def copula_conditional_probability(
    copula_model: str,
    x_hist: np.ndarray,
    y_hist: np.ndarray,
    u_i: float,
    u_j: float,
    student_t_df: int,
    mixture_weights: tuple[float, float, float],
    estimate_mixture_weights: bool = False,
    mixture_weight_floor: float = 0.05,
    mixture_weight_max_iter: int = 50,
    mixture_weight_tol: float = 1e-6,
) -> tuple[float, dict[str, float]]:
    copula_model = canonical_copula_model(copula_model)
    if copula_model != "mixture":
        raise ValueError(f"Unsupported copula_model={copula_model!r}. Use one of {sorted(SUPPORTED_COPULA_MODELS)}.")

    params = estimate_mixture_component_params(
        x_hist=x_hist,
        y_hist=y_hist,
        student_t_df=student_t_df,
        mixture_weights=mixture_weights,
        estimate_mixture_weights=estimate_mixture_weights,
        mixture_weight_floor=mixture_weight_floor,
        mixture_weight_max_iter=mixture_weight_max_iter,
        mixture_weight_tol=mixture_weight_tol,
    )
    probabilities = mixture_conditional_probabilities(
        u_i=u_i,
        u_j=u_j,
        params=params,
        student_t_df=student_t_df,
    )
    return probabilities["mi_i_given_j"], {**params, **probabilities}


def rolling_copula_pair_signals(
    returns: pd.DataFrame,
    pairs: pd.DataFrame,
    window: int,
    copula_model: str = "mixture",
    student_t_df: int = 6,
    mixture_weights: tuple[float, float, float] = (1.0 / 3.0, 1.0 / 3.0, 1.0 / 3.0),
    estimate_mixture_weights: bool = False,
    mixture_weight_floor: float = 0.05,
    mixture_weight_max_iter: int = 50,
    mixture_weight_tol: float = 1e-6,
    output_path: Path | None = RESULTS_DIR / "pair_signals.csv",
) -> pd.DataFrame:
    copula_model = canonical_copula_model(copula_model)
    if copula_model not in SUPPORTED_COPULA_MODELS:
        raise ValueError(f"Unsupported copula_model={copula_model!r}. Use one of {sorted(SUPPORTED_COPULA_MODELS)}.")
    if student_t_df <= 2:
        raise ValueError("student_t_df must be greater than 2.")

    signal_records: list[dict[str, object]] = []

    for row in pairs.itertuples(index=False):
        asset_i = row.asset_i
        asset_j = row.asset_j
        pair_returns = returns[[asset_i, asset_j]].dropna()
        values_i = pair_returns[asset_i].to_numpy()
        values_j = pair_returns[asset_j].to_numpy()
        dates = pair_returns.index

        for pos in range(window, len(pair_returns)):
            x_hist = values_i[pos - window:pos]
            y_hist = values_j[pos - window:pos]
            x_now = values_i[pos]
            y_now = values_j[pos]

            u_i = empirical_u_current(x_hist, x_now)
            u_j = empirical_u_current(y_hist, y_now)
            params = estimate_mixture_component_params(
                x_hist=x_hist,
                y_hist=y_hist,
                student_t_df=student_t_df,
                mixture_weights=mixture_weights,
                estimate_mixture_weights=estimate_mixture_weights,
                mixture_weight_floor=mixture_weight_floor,
                mixture_weight_max_iter=mixture_weight_max_iter,
                mixture_weight_tol=mixture_weight_tol,
            )
            probabilities = mixture_conditional_probabilities(
                u_i=u_i,
                u_j=u_j,
                params=params,
                student_t_df=student_t_df,
            )
            mi_i_given_j = probabilities["mi_i_given_j"]
            mi_j_given_i = probabilities["mi_j_given_i"]
            bidirectional_score = 0.5 * (mi_i_given_j - mi_j_given_i)
            spread_vol = float(np.std(x_hist - y_hist, ddof=1))

            signal_records.append(
                {
                    "date": dates[pos],
                    "asset_i": asset_i,
                    "asset_j": asset_j,
                    "copula_model": copula_model,
                    **params,
                    **probabilities,
                    "u_i": u_i,
                    "u_j": u_j,
                    "conditional_prob": mi_i_given_j,
                    "raw_i_score": mi_i_given_j - 0.5,
                    "raw_j_score": mi_j_given_i - 0.5,
                    "bidirectional_score": bidirectional_score,
                    "mispricing_score": bidirectional_score,
                    "spread_vol": spread_vol,
                }
            )

    signals = pd.DataFrame(signal_records)
    if not signals.empty:
        signals = signals.sort_values(["date", "asset_i", "asset_j"]).reset_index(drop=True)
    if output_path is not None:
        signals.to_csv(output_path, index=False)
    return signals


## Portfolio construction e backtest

Questa versione mantiene il segnale copula ma semplifica il trading layer.

La logica e:

1. calcola soglie di entry ed exit per ogni coppia usando solo il train;
2. stima l'edge storico next-day della coppia, sempre solo sul train;
3. entra solo quando $|S_{ij,t}|$ supera la soglia alta della coppia e l'edge storico stimato batte una soglia minima;
4. esce quando lo score torna vicino alla normalita, cambia segno o supera il massimo holding period;
5. usa massimo `top_n_pairs` coppie attive;
6. assegna lo stesso gross slot a ogni coppia, senza volatility sizing.

Le soglie per coppia sono:

$$
q_{ij}^{entry} = \max\left(q_{min}, Q_{\alpha}\left(|S_{ij,t}|\right)\right)
$$

$$
q_{ij}^{exit} = Q_{\beta}\left(|S_{ij,t}|\right)
$$

L'expected edge viene stimato guardando solo i segnali train che avrebbero superato l'entry threshold:

$$
\widehat{edge}_{ij} = \mathbb{E}_{train}\left[-\frac{1}{2}\operatorname{sign}(S_{ij,t})(r_{i,t+1}-r_{j,t+1}) \mid |S_{ij,t}| \ge q_{ij}^{entry}\right]
$$

La coppia e tradabile solo se:

$$
10000 \times \widehat{edge}_{ij} \ge edge_{min}
$$

Il sizing e equal-slot. Ogni coppia usa al massimo uno slot di gross exposure:

$$
w_{pair} = \frac{1}{N_{max}}
$$

La regola di PnL evita look-ahead bias:

$$
\operatorname{PnL}_{t} = \sum_i w_{i,t-1} r_{i,t}
$$

I costi sono proporzionali al turnover:

$$
\operatorname{Cost}_{t} = \operatorname{Turnover}_{t} \times \frac{\operatorname{cost\_bps}}{10000}
$$

$$
R^{net}_{t} = R^{gross}_{t} - \operatorname{Cost}_{t}
$$


In [ ]:
def _pair_key(asset_i: str, asset_j: str) -> str:
    return f"{asset_i}|{asset_j}"


def _add_pair_key(signals: pd.DataFrame) -> pd.DataFrame:
    signals = signals.copy()
    signals["pair_key"] = signals["asset_i"].astype(str) + "|" + signals["asset_j"].astype(str)
    return signals


def _signal_score_column(signals: pd.DataFrame) -> str:
    return "bidirectional_score" if "bidirectional_score" in signals.columns else "mispricing_score"


def compute_pair_signal_thresholds(
    signals: pd.DataFrame,
    entry_quantile: float,
    exit_quantile: float,
    min_entry_score: float,
    train_end: str | None,
) -> pd.DataFrame:
    if not 0 < exit_quantile < entry_quantile < 1:
        raise ValueError("Use 0 < exit_quantile < entry_quantile < 1.")
    if min_entry_score < 0:
        raise ValueError("min_entry_score must be non-negative.")

    score_col = _signal_score_column(signals)
    threshold_input = _add_pair_key(signals)
    threshold_input["date"] = pd.to_datetime(threshold_input["date"])
    threshold_input[score_col] = pd.to_numeric(threshold_input[score_col], errors="coerce")
    threshold_input = threshold_input.dropna(subset=[score_col])

    if train_end is not None:
        train_slice = threshold_input.loc[threshold_input["date"] <= pd.Timestamp(train_end)]
    else:
        train_slice = threshold_input
    if train_slice.empty:
        train_slice = threshold_input

    rows: list[dict[str, object]] = []
    for pair_key, group in train_slice.groupby("pair_key"):
        abs_score = group[score_col].abs()
        entry_threshold = abs_score.quantile(entry_quantile)
        exit_threshold = abs_score.quantile(exit_quantile)
        if pd.isna(entry_threshold):
            entry_threshold = min_entry_score
        if pd.isna(exit_threshold):
            exit_threshold = min_entry_score * 0.5
        entry_threshold = float(max(min_entry_score, entry_threshold))
        exit_threshold = float(min(exit_threshold, entry_threshold * 0.75))
        rows.append(
            {
                "pair_key": pair_key,
                "entry_threshold": entry_threshold,
                "exit_threshold": exit_threshold,
            }
        )

    return pd.DataFrame(rows).set_index("pair_key")


def compute_pair_expected_edges(
    signals: pd.DataFrame,
    returns: pd.DataFrame,
    thresholds: pd.DataFrame,
    train_end: str | None,
    cost_bps: float,
    expected_edge_cost_multiplier: float,
    min_expected_edge_bps: float,
    min_expected_edge_observations: int,
    output_path: Path | None = RESULTS_DIR / "pair_expected_edges.csv",
) -> pd.DataFrame:
    if min_expected_edge_observations < 1:
        raise ValueError("min_expected_edge_observations must be at least 1.")
    if expected_edge_cost_multiplier < 0:
        raise ValueError("expected_edge_cost_multiplier must be non-negative.")

    score_col = _signal_score_column(signals)
    prepared = _add_pair_key(signals)
    prepared["date"] = pd.to_datetime(prepared["date"])
    prepared["_score"] = pd.to_numeric(prepared[score_col], errors="coerce")
    prepared["_abs_score"] = prepared["_score"].abs()
    prepared = prepared.dropna(subset=["date", "_score"])

    if train_end is not None:
        prepared = prepared.loc[prepared["date"] <= pd.Timestamp(train_end)]

    forward_returns = returns.shift(-1)
    records: list[dict[str, object]] = []
    threshold_map = thresholds["entry_threshold"].to_dict()

    for _, signal in prepared.iterrows():
        pair_key = str(signal["pair_key"])
        entry_threshold = threshold_map.get(pair_key)
        if entry_threshold is None or float(signal["_abs_score"]) < float(entry_threshold):
            continue

        date = pd.Timestamp(signal["date"])
        if date not in forward_returns.index:
            continue

        asset_i = str(signal["asset_i"])
        asset_j = str(signal["asset_j"])
        next_i = forward_returns.at[date, asset_i]
        next_j = forward_returns.at[date, asset_j]
        score = float(signal["_score"])
        if pd.isna(next_i) or pd.isna(next_j) or score == 0:
            continue

        pair_return = -0.5 * np.sign(score) * (float(next_i) - float(next_j))
        records.append(
            {
                "pair_key": pair_key,
                "date": date,
                "pair_forward_return": pair_return,
            }
        )

    minimum_edge_bps = max(float(min_expected_edge_bps), float(cost_bps) * float(expected_edge_cost_multiplier))
    edge_rows: list[dict[str, object]] = []
    edge_input = pd.DataFrame(records)

    for pair_key in thresholds.index:
        if edge_input.empty or pair_key not in set(edge_input["pair_key"]):
            n_obs = 0
            expected_edge = np.nan
            edge_hit_rate = np.nan
        else:
            group = edge_input.loc[edge_input["pair_key"] == pair_key, "pair_forward_return"]
            n_obs = int(len(group))
            expected_edge = float(group.mean())
            edge_hit_rate = float((group > 0).mean())

        expected_edge_bps = expected_edge * 10000.0 if np.isfinite(expected_edge) else np.nan
        edge_pass = bool(
            n_obs >= min_expected_edge_observations
            and np.isfinite(expected_edge_bps)
            and expected_edge_bps >= minimum_edge_bps
        )
        edge_rows.append(
            {
                "pair_key": pair_key,
                "edge_observations": n_obs,
                "expected_edge": expected_edge,
                "expected_edge_bps": expected_edge_bps,
                "edge_hit_rate": edge_hit_rate,
                "minimum_edge_bps": minimum_edge_bps,
                "edge_pass": edge_pass,
            }
        )

    edges = pd.DataFrame(edge_rows).set_index("pair_key")
    if output_path is not None:
        edges.reset_index().to_csv(output_path, index=False)
    return edges


def build_positions_from_pair_signals(
    signals: pd.DataFrame,
    returns: pd.DataFrame,
    top_n_pairs: int,
    entry_quantile: float,
    exit_quantile: float,
    min_entry_score: float,
    max_holding_days: int,
    rebalance_every_n_days: int = 1,
    max_daily_turnover: float | None = None,
    train_end: str | None = None,
    cost_bps: float = 0.0,
    expected_edge_cost_multiplier: float = 0.0,
    min_expected_edge_bps: float = 0.0,
    min_expected_edge_observations: int = 20,
    output_path: Path | None = RESULTS_DIR / "daily_positions.csv",
) -> pd.DataFrame:
    positions = pd.DataFrame(0.0, index=returns.index, columns=returns.columns)
    if signals.empty:
        return positions

    if top_n_pairs < 1:
        raise ValueError("top_n_pairs must be at least 1.")
    if max_holding_days < 1:
        raise ValueError("max_holding_days must be at least 1.")
    if rebalance_every_n_days < 1:
        raise ValueError("rebalance_every_n_days must be at least 1.")
    if max_daily_turnover is not None and max_daily_turnover <= 0:
        raise ValueError("max_daily_turnover must be positive or None.")

    score_col = _signal_score_column(signals)
    prepared = _add_pair_key(signals)
    prepared["date"] = pd.to_datetime(prepared["date"])
    prepared["_score"] = pd.to_numeric(prepared[score_col], errors="coerce")
    prepared["_abs_score"] = prepared["_score"].abs()
    prepared = prepared.dropna(subset=["date", "_score"])

    thresholds = compute_pair_signal_thresholds(
        signals=prepared,
        entry_quantile=entry_quantile,
        exit_quantile=exit_quantile,
        min_entry_score=min_entry_score,
        train_end=train_end,
    )
    edge_estimates = compute_pair_expected_edges(
        signals=prepared,
        returns=returns,
        thresholds=thresholds,
        train_end=train_end,
        cost_bps=cost_bps,
        expected_edge_cost_multiplier=expected_edge_cost_multiplier,
        min_expected_edge_bps=min_expected_edge_bps,
        min_expected_edge_observations=min_expected_edge_observations,
    )

    signals_by_date = {
        pd.Timestamp(date): group.set_index("pair_key", drop=False)
        for date, group in prepared.groupby("date")
    }

    active_trades: dict[str, dict[str, object]] = {}

    def get_signal_for_pair(day_signals: pd.DataFrame | None, pair_key: str) -> pd.Series | None:
        if day_signals is None or pair_key not in day_signals.index:
            return None
        pair_signal = day_signals.loc[pair_key]
        if isinstance(pair_signal, pd.DataFrame):
            pair_signal = pair_signal.iloc[0]
        return pair_signal

    def pair_exit_threshold(pair_key: str) -> float:
        if pair_key in thresholds.index:
            return float(thresholds.loc[pair_key, "exit_threshold"])
        return float(min_entry_score * 0.5)

    def should_exit(pair_key: str, trade: dict[str, object], day_signals: pd.DataFrame | None) -> bool:
        pair_signal = get_signal_for_pair(day_signals, pair_key)
        if pair_signal is None:
            return True

        score = float(pair_signal["_score"])
        if not np.isfinite(score):
            return True

        abs_score = abs(score)
        current_direction = 1 if score > 0 else -1 if score < 0 else 0
        original_direction = int(trade["direction"])
        holding_days = int(trade["holding_days"])

        if abs_score <= pair_exit_threshold(pair_key):
            return True
        if current_direction != 0 and current_direction != original_direction:
            return True
        if holding_days >= max_holding_days:
            return True
        return False

    def target_position_from_active(day_signals: pd.DataFrame | None) -> pd.Series:
        target = pd.Series(0.0, index=returns.columns)
        if day_signals is None or not active_trades:
            return target

        pair_gross_slot = 1.0 / top_n_pairs
        leg_weight = 0.5 * pair_gross_slot

        for pair_key, trade in active_trades.items():
            pair_signal = get_signal_for_pair(day_signals, pair_key)
            if pair_signal is None:
                continue

            asset_i = str(pair_signal["asset_i"])
            asset_j = str(pair_signal["asset_j"])
            direction = int(trade["direction"])

            if direction > 0:
                # Positive score: asset i rich, asset j cheap.
                target.loc[asset_i] -= leg_weight
                target.loc[asset_j] += leg_weight
            else:
                # Negative score: asset i cheap, asset j rich.
                target.loc[asset_i] += leg_weight
                target.loc[asset_j] -= leg_weight

        gross = target.abs().sum()
        if gross > 1.0:
            target = target / gross
        return target

    previous_position = pd.Series(0.0, index=returns.columns)
    for idx, date in enumerate(returns.index):
        date = pd.Timestamp(date)
        day_signals = signals_by_date.get(date)
        exited_pairs_today: set[str] = set()

        for trade in active_trades.values():
            trade["holding_days"] = int(trade["holding_days"]) + 1

        if day_signals is not None:
            for pair_key in list(active_trades):
                if should_exit(pair_key, active_trades[pair_key], day_signals):
                    del active_trades[pair_key]
                    exited_pairs_today.add(pair_key)

        is_rebalance_day = idx % rebalance_every_n_days == 0
        if is_rebalance_day and day_signals is not None:
            available_slots = max(0, top_n_pairs - len(active_trades))
            if available_slots > 0:
                candidates = day_signals.loc[
                    ~day_signals["pair_key"].isin(active_trades)
                    & ~day_signals["pair_key"].isin(exited_pairs_today)
                ].copy()
                candidates["entry_threshold"] = candidates["pair_key"].map(thresholds["entry_threshold"]).fillna(min_entry_score)
                candidates["expected_edge_bps"] = candidates["pair_key"].map(edge_estimates["expected_edge_bps"])
                candidates["edge_pass"] = candidates["pair_key"].map(edge_estimates["edge_pass"]).fillna(False)
                candidates = candidates.loc[
                    (candidates["_score"] != 0)
                    & (candidates["_abs_score"] >= candidates["entry_threshold"])
                    & (candidates["edge_pass"])
                ].copy()
                candidates["rank_score"] = candidates["_abs_score"] * candidates["expected_edge_bps"].clip(lower=0)
                candidates = candidates.sort_values("rank_score", ascending=False).head(available_slots)

                for _, pair_signal in candidates.iterrows():
                    score = float(pair_signal["_score"])
                    active_trades[str(pair_signal["pair_key"])] = {
                        "direction": 1 if score > 0 else -1,
                        "holding_days": 0,
                        "entry_abs_score": abs(score),
                    }

        if day_signals is not None:
            target_position = target_position_from_active(day_signals)
        else:
            target_position = previous_position.copy()

        trade = target_position - previous_position
        day_turnover = trade.abs().sum()
        if max_daily_turnover is not None and day_turnover > max_daily_turnover:
            trade = trade * (max_daily_turnover / day_turnover)
            target_position = previous_position + trade

        positions.loc[date] = target_position
        previous_position = target_position

    if output_path is not None:
        positions.to_csv(output_path)
    return positions


def run_backtest(
    positions: pd.DataFrame,
    returns: pd.DataFrame,
    cost_bps: float,
    output_path: Path | None = RESULTS_DIR / "backtest.csv",
) -> pd.DataFrame:
    positions = positions.reindex(returns.index).fillna(0.0)
    positions_for_pnl = positions.shift(1).fillna(0.0)

    gross_return = (positions_for_pnl * returns).sum(axis=1)
    turnover = positions.diff().abs().sum(axis=1).fillna(positions.abs().sum(axis=1))
    transaction_cost = turnover * cost_bps / 10000.0
    net_return = gross_return - transaction_cost

    backtest = pd.DataFrame(
        {
            "gross_return": gross_return,
            "turnover": turnover,
            "transaction_cost": transaction_cost,
            "net_return": net_return,
            "gross_exposure": positions.abs().sum(axis=1),
            "net_exposure": positions.sum(axis=1),
        }
    )
    backtest["gross_equity"] = (1.0 + backtest["gross_return"]).cumprod()
    backtest["net_equity"] = (1.0 + backtest["net_return"]).cumprod()
    if output_path is not None:
        backtest.to_csv(output_path)
    return backtest


## Metrics e IC

Oltre alle metriche di performance, misuriamo se il segnale ha contenuto predittivo cross-sectionally.

Per ogni coppia, se $s_{i \mid j,t}$ e positivo, il trade atteso e short $i$, long $j$. Quindi il target a un giorno e:

$$
y_{i,j,t+1} = -\left(r_{i,t+1} - r_{j,t+1}\right)
$$

L'information coefficient giornaliero e:

$$
IC_t = \operatorname{corr}\left(s_{i \mid j,t}, y_{i,j,t+1}\right)
$$

Il rank IC usa Spearman correlation al posto di Pearson.

In [ ]:
def max_drawdown(equity: pd.Series) -> float:
    running_max = equity.cummax()
    drawdown = equity / running_max - 1.0
    return float(drawdown.min())


def performance_stats(
    backtest: pd.DataFrame,
    returns: pd.DataFrame,
    strategy_col: str = "net_return",
) -> dict[str, float]:
    strategy_returns = backtest[strategy_col].dropna()
    if strategy_returns.empty:
        return {}

    n_days = len(strategy_returns)
    equity = (1.0 + strategy_returns).cumprod()
    ann_return = float(equity.iloc[-1] ** (252.0 / n_days) - 1.0)
    ann_vol = float(strategy_returns.std(ddof=1) * math.sqrt(252.0))
    sharpe = float(strategy_returns.mean() / strategy_returns.std(ddof=1) * math.sqrt(252.0)) if strategy_returns.std(ddof=1) > 0 else np.nan

    downside = strategy_returns[strategy_returns < 0]
    sortino = float(strategy_returns.mean() / downside.std(ddof=1) * math.sqrt(252.0)) if len(downside) > 1 and downside.std(ddof=1) > 0 else np.nan
    mdd = max_drawdown(equity)
    calmar = float(ann_return / abs(mdd)) if mdd < 0 else np.nan

    beta = np.nan
    if "SPY" in returns.columns:
        aligned = pd.concat([strategy_returns.rename("strategy"), returns["SPY"].rename("SPY")], axis=1).dropna()
        if len(aligned) > 2 and aligned["SPY"].var() > 0:
            beta = float(aligned.cov().loc["strategy", "SPY"] / aligned["SPY"].var())

    return {
        "annualized_return": ann_return,
        "annualized_volatility": ann_vol,
        "sharpe_ratio": sharpe,
        "sortino_ratio": sortino,
        "calmar_ratio": calmar,
        "max_drawdown": mdd,
        "hit_rate": float((strategy_returns > 0).mean()),
        "average_daily_turnover": float(backtest["turnover"].mean()),
        "average_transaction_cost": float(backtest["transaction_cost"].mean()),
        "average_gross_exposure": float(backtest["gross_exposure"].mean()),
        "average_net_exposure": float(backtest["net_exposure"].mean()),
        "market_beta_spy": beta,
    }


def compute_pair_ic(
    signals: pd.DataFrame,
    returns: pd.DataFrame,
    output_path: Path | None = RESULTS_DIR / "daily_ic.csv",
) -> tuple[pd.DataFrame, dict[str, float]]:
    if signals.empty:
        return pd.DataFrame(), {}

    forward_returns = returns.shift(-1)
    records: list[dict[str, object]] = []

    for signal in signals.itertuples(index=False):
        date = pd.Timestamp(signal.date)
        if date not in forward_returns.index:
            continue
        next_i = forward_returns.loc[date, signal.asset_i]
        next_j = forward_returns.loc[date, signal.asset_j]
        if pd.isna(next_i) or pd.isna(next_j):
            continue
        records.append(
            {
                "date": date,
                "mispricing_score": signal.mispricing_score,
                "forward_pair_alpha": -(next_i - next_j),
            }
        )

    ic_input = pd.DataFrame(records)
    daily_records: list[dict[str, float | pd.Timestamp]] = []
    for date, group in ic_input.groupby("date"):
        if len(group) < 3 or group["mispricing_score"].std(ddof=1) == 0 or group["forward_pair_alpha"].std(ddof=1) == 0:
            continue
        ic = group["mispricing_score"].corr(group["forward_pair_alpha"])
        rank_ic = spearmanr(group["mispricing_score"], group["forward_pair_alpha"]).correlation
        daily_records.append({"date": date, "IC": float(ic), "rank_IC": float(rank_ic)})

    if not daily_records:
        return pd.DataFrame(), {
            "average_IC": np.nan,
            "average_rank_IC": np.nan,
            "IC_t_stat": np.nan,
            "rank_IC_t_stat": np.nan,
        }

    daily_ic = pd.DataFrame(daily_records).set_index("date").sort_index()
    summary = {
        "average_IC": float(daily_ic["IC"].mean()),
        "average_rank_IC": float(daily_ic["rank_IC"].mean()),
        "IC_t_stat": float(daily_ic["IC"].mean() / daily_ic["IC"].std(ddof=1) * math.sqrt(len(daily_ic))) if len(daily_ic) > 2 and daily_ic["IC"].std(ddof=1) > 0 else np.nan,
        "rank_IC_t_stat": float(daily_ic["rank_IC"].mean() / daily_ic["rank_IC"].std(ddof=1) * math.sqrt(len(daily_ic))) if len(daily_ic) > 2 and daily_ic["rank_IC"].std(ddof=1) > 0 else np.nan,
    }
    if output_path is not None:
        daily_ic.to_csv(output_path)
    return daily_ic, summary


def format_metrics(metrics: dict[str, float]) -> pd.DataFrame:
    return pd.DataFrame(metrics, index=["value"]).T

## Esecuzione MVP

Eseguire la cella seguente per lanciare il workflow completo. La prima esecuzione scarica i dati; le successive usano `data/processed/prices.csv` salvo `REFRESH_DATA = True`.

In [ ]:
REFRESH_DATA = False

prices = load_or_download_prices(refresh=REFRESH_DATA)
returns = compute_log_returns(prices)
candidate_pairs = select_candidate_pairs(
    returns=returns,
    train_end=CONFIG.train_end,
    top_k=CONFIG.peers_per_asset,
    max_pairs=CONFIG.max_candidate_pairs,
    excluded_assets=CONFIG.excluded_assets,
    blocked_overlap_groups=CONFIG.blocked_overlap_groups,
    min_abs_corr=CONFIG.min_abs_corr,
    max_spread_half_life=CONFIG.max_spread_half_life,
    min_spread_vol=CONFIG.min_spread_vol,
)
pair_signals = rolling_copula_pair_signals(
    returns=returns,
    pairs=candidate_pairs,
    window=CONFIG.rolling_window,
    copula_model=CONFIG.copula_model,
    student_t_df=CONFIG.student_t_df,
    mixture_weights=CONFIG.mixture_weights,
    estimate_mixture_weights=CONFIG.estimate_mixture_weights,
    mixture_weight_floor=CONFIG.mixture_weight_floor,
    mixture_weight_max_iter=CONFIG.mixture_weight_max_iter,
    mixture_weight_tol=CONFIG.mixture_weight_tol,
)

positions = build_positions_from_pair_signals(
    signals=pair_signals,
    returns=returns,
    top_n_pairs=CONFIG.top_n_pairs,
    entry_quantile=CONFIG.entry_quantile,
    exit_quantile=CONFIG.exit_quantile,
    min_entry_score=CONFIG.min_entry_score,
    max_holding_days=CONFIG.max_holding_days,
    rebalance_every_n_days=CONFIG.rebalance_every_n_days,
    max_daily_turnover=CONFIG.max_daily_turnover,
    train_end=CONFIG.train_end,
    cost_bps=CONFIG.cost_bps,
    expected_edge_cost_multiplier=CONFIG.expected_edge_cost_multiplier,
    min_expected_edge_bps=CONFIG.min_expected_edge_bps,
    min_expected_edge_observations=CONFIG.min_expected_edge_observations,
)
backtest = run_backtest(positions=positions, returns=returns, cost_bps=CONFIG.cost_bps)
daily_ic, ic_summary = compute_pair_ic(pair_signals, returns)

net_stats = performance_stats(backtest, returns, strategy_col="net_return")
gross_stats = performance_stats(backtest, returns, strategy_col="gross_return")
gross_summary = {
    f"gross_{key}": value
    for key, value in gross_stats.items()
    if key in {
        "annualized_return",
        "annualized_volatility",
        "sharpe_ratio",
        "sortino_ratio",
        "calmar_ratio",
        "max_drawdown",
        "hit_rate",
    }
}
summary = {**gross_summary, **net_stats, **ic_summary}
summary_df = format_metrics(summary)
summary_df.to_csv(RESULTS_DIR / "performance_summary.csv")

edge_estimates = pd.read_csv(RESULTS_DIR / "pair_expected_edges.csv")

print(f"Assets after cleaning: {returns.shape[1]}")
print(f"Excluded assets: {CONFIG.excluded_assets}")
print(f"Copula model: {CONFIG.copula_model}")
print(f"Initial mixture weights G/T/Clayton: {normalize_mixture_weights(CONFIG.mixture_weights)}")
print(f"Estimate mixture weights: {CONFIG.estimate_mixture_weights}")
print(f"Mixture weight floor: {CONFIG.mixture_weight_floor}")
print(f"Daily observations: {returns.shape[0]}")
print(f"Candidate pairs: {len(candidate_pairs)}")
print(f"Pair signal rows: {len(pair_signals)}")
print(f"Tradable pairs after edge filter: {int(edge_estimates['edge_pass'].sum())}")
print(f"Top N active pairs: {CONFIG.top_n_pairs}")
print(f"Entry quantile: {CONFIG.entry_quantile}")
print(f"Exit quantile: {CONFIG.exit_quantile}")
print(f"Minimum entry score: {CONFIG.min_entry_score}")
print(f"Minimum expected edge bps: {CONFIG.min_expected_edge_bps}")
print(f"Expected edge cost multiplier: {CONFIG.expected_edge_cost_multiplier}")
print(f"Max holding days: {CONFIG.max_holding_days}")
print(f"Rebalance every N days: {CONFIG.rebalance_every_n_days}")
print(f"Max daily turnover: {CONFIG.max_daily_turnover}")
display(candidate_pairs.head(10))
display(edge_estimates.sort_values("expected_edge_bps", ascending=False).head(10))
display(summary_df)


## Performance by period

Il test set inizia il 2023-01-01 e non deve essere usato per ottimizzare soglie, finestre o numero di pair trades.

In [ ]:
def period_slice(backtest: pd.DataFrame, start: str | None, end: str | None) -> pd.DataFrame:
    period = backtest.copy()
    if start is not None:
        period = period.loc[start:]
    if end is not None:
        period = period.loc[:end]
    return period


periods = {
    "train": (CONFIG.start_date, CONFIG.train_end),
    "validation": (CONFIG.validation_start, CONFIG.validation_end),
    "test": (CONFIG.test_start, None),
}

period_rows = []
for period_name, (start, end) in periods.items():
    bt_period = period_slice(backtest, start, end)
    ret_period = period_slice(returns, start, end)
    if len(bt_period) == 0:
        continue
    row = performance_stats(bt_period, ret_period, strategy_col="net_return")
    row["period"] = period_name
    row["start"] = str(bt_period.index.min().date())
    row["end"] = str(bt_period.index.max().date())
    period_rows.append(row)

performance_by_period = pd.DataFrame(period_rows).set_index("period")
performance_by_period.to_csv(RESULTS_DIR / "performance_by_period.csv")
display(performance_by_period)

## Plot diagnostici

I plot vengono salvati in `results/plots/`.

In [ ]:
def plot_equity_curve(backtest: pd.DataFrame) -> None:
    fig, ax = plt.subplots(figsize=(12, 5))
    backtest[["gross_equity", "net_equity"]].plot(ax=ax, linewidth=1.5)
    model_label = CONFIG.copula_model.replace("_", "-").title()
    ax.set_title(f"{model_label} Copula MVP - Equity Curve")
    ax.set_ylabel("Growth of $1")
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / "equity_curve.png", dpi=150)
    plt.show()


def plot_drawdown(backtest: pd.DataFrame) -> None:
    net_equity = backtest["net_equity"]
    drawdown = net_equity / net_equity.cummax() - 1.0
    fig, ax = plt.subplots(figsize=(12, 4))
    drawdown.plot(ax=ax, color="firebrick", linewidth=1.2)
    ax.set_title("Net Drawdown")
    ax.set_ylabel("Drawdown")
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / "drawdown.png", dpi=150)
    plt.show()


def plot_turnover(backtest: pd.DataFrame) -> None:
    fig, ax = plt.subplots(figsize=(12, 4))
    backtest["turnover"].rolling(21).mean().plot(ax=ax, color="steelblue", linewidth=1.2)
    ax.set_title("21-Day Average Turnover")
    ax.set_ylabel("Turnover")
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / "turnover.png", dpi=150)
    plt.show()


plot_equity_curve(backtest)
plot_drawdown(backtest)
plot_turnover(backtest)

## Robustness starter

La griglia sotto e volutamente piccola. Serve per iniziare a capire se il segnale sopravvive a variazioni ragionevoli dei parametri.

Ora la robustezza varia:

- severita della pair selection;
- percentile di entry;
- expected edge minimo;
- costo implicito richiesto dal filtro edge.

Per un report serio, ampliare la griglia e non scegliere la configurazione guardando il test set.


In [ ]:
def run_small_robustness_grid(
    returns: pd.DataFrame,
    candidate_pairs: pd.DataFrame,
    copula_models: list[str] = ["mixture"],
    windows: list[int] = [126, 252],
    entry_quantiles: list[float] = [0.90, 0.95],
    exit_quantiles: list[float] = [0.50],
    min_entry_scores: list[float] = [0.20, 0.25, 0.30],
    min_expected_edge_bps_values: list[float] = [0.0, 1.0, 2.0],
    top_n_values: list[int] = [1, 3],
    max_holding_values: list[int] = [1, 3],
    rebalance_values: list[int] = [5],
    max_turnover_values: list[float | None] = [0.35],
    costs_bps: list[float] = [0.0, 5.0],
) -> pd.DataFrame:
    rows: list[dict[str, object]] = []
    signal_cache: dict[tuple[str, int], pd.DataFrame] = {}

    for model in copula_models:
        for window in windows:
            cache_key = (model, window)
            if cache_key not in signal_cache:
                signal_cache[cache_key] = rolling_copula_pair_signals(
                    returns=returns,
                    pairs=candidate_pairs,
                    window=window,
                    copula_model=model,
                    student_t_df=CONFIG.student_t_df,
                    mixture_weights=CONFIG.mixture_weights,
                    estimate_mixture_weights=CONFIG.estimate_mixture_weights,
                    mixture_weight_floor=CONFIG.mixture_weight_floor,
                    mixture_weight_max_iter=CONFIG.mixture_weight_max_iter,
                    mixture_weight_tol=CONFIG.mixture_weight_tol,
                    output_path=None,
                )
            signals_for_window = signal_cache[cache_key]

            for entry_quantile in entry_quantiles:
                for exit_quantile in exit_quantiles:
                    if exit_quantile >= entry_quantile:
                        continue
                    for min_entry_score in min_entry_scores:
                        for min_expected_edge_bps in min_expected_edge_bps_values:
                            for top_n in top_n_values:
                                for max_holding_days in max_holding_values:
                                    for rebalance_every_n_days in rebalance_values:
                                        for max_daily_turnover in max_turnover_values:
                                            for cost_bps in costs_bps:
                                                positions_grid = build_positions_from_pair_signals(
                                                    signals=signals_for_window,
                                                    returns=returns,
                                                    top_n_pairs=top_n,
                                                    entry_quantile=entry_quantile,
                                                    exit_quantile=exit_quantile,
                                                    min_entry_score=min_entry_score,
                                                    max_holding_days=max_holding_days,
                                                    rebalance_every_n_days=rebalance_every_n_days,
                                                    max_daily_turnover=max_daily_turnover,
                                                    train_end=CONFIG.train_end,
                                                    cost_bps=cost_bps,
                                                    expected_edge_cost_multiplier=CONFIG.expected_edge_cost_multiplier,
                                                    min_expected_edge_bps=min_expected_edge_bps,
                                                    min_expected_edge_observations=CONFIG.min_expected_edge_observations,
                                                    output_path=None,
                                                )
                                                bt_grid = run_backtest(positions_grid, returns, cost_bps=cost_bps, output_path=None)
                                                metrics = performance_stats(
                                                    bt_grid.loc[CONFIG.validation_start:CONFIG.validation_end],
                                                    returns.loc[CONFIG.validation_start:CONFIG.validation_end],
                                                )
                                                rows.append(
                                                    {
                                                        "copula_model": model,
                                                        "window": window,
                                                        "entry_quantile": entry_quantile,
                                                        "exit_quantile": exit_quantile,
                                                        "min_entry_score": min_entry_score,
                                                        "min_expected_edge_bps": min_expected_edge_bps,
                                                        "top_n_pairs": top_n,
                                                        "max_holding_days": max_holding_days,
                                                        "rebalance_every_n_days": rebalance_every_n_days,
                                                        "max_daily_turnover": max_daily_turnover,
                                                        "cost_bps": cost_bps,
                                                        **metrics,
                                                    }
                                                )

    robustness = pd.DataFrame(rows)
    robustness.to_csv(RESULTS_DIR / "robustness_table.csv", index=False)
    return robustness


RUN_ROBUSTNESS = False

if RUN_ROBUSTNESS:
    robustness_table = run_small_robustness_grid(returns, candidate_pairs)
    display(robustness_table.sort_values("sharpe_ratio", ascending=False).head(10))


## Interpretazione research

Dopo l'esecuzione, leggere i risultati con questa sequenza:

1. Quante coppie sopravvivono alla pair selection v2?
2. Quante coppie passano anche l'expected edge filter?
3. Il segnale ha average IC positivo usando `bidirectional_score`?
4. La performance gross e gia positiva prima dei costi, e quanto viene mangiata dal net?
5. Il test set e coerente con validation, o il risultato e concentrato in train?
6. Turnover e drawdown sono compatibili con una strategia realisticamente tradabile?

Questa versione separa tre problemi diversi: qualita delle coppie, qualita del segnale e costo di monetizzazione. Se il filtro edge lascia poche coppie o il net resta negativo, il problema non si risolve aggiungendo copule.

Estensioni naturali, solo dopo un segnale base stabile:

- confrontare pesi mixture stimati vs pesi fissi 1/3;
- stimare il costo effettivo per ETF invece di un costo unico;
- usare beta-neutral sizing sulle poche coppie che passano l'edge filter;
- sector-neutral pair selection;
- confronto con una baseline z-score sullo stesso pair universe.
